In [56]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED) #为CPU上的PyTorch随机数生成器设定种子。这会影响模型参数初始化、训练时的随机采样等。

if torch.cuda.is_available(): # 先检查是否有可用的CUDA（NVIDIA GPU）
    torch.cuda.manual_seed(SEED) 
    torch.cuda.manual_seed_all(SEED) 

torch.backends.cudnn.deterministic = True  #强制cuDNN使用确定性的卷积算法。
torch.backends.cudnn.benchmark = False  #关闭cuDNN的自动基准测试

# 它为你搭建了一个可复现的实验环境。这样做的好处是：

#当你和其他人跑同一个代码时，如果所有随机种子都固定，理论上会得到完全一样的模型训练结果。

#当你修改代码的某一部分后，可以确定看到的效果差异完全来自代码改动本身，而不是随机运气。

# 虽然关闭benchmark可能会让训练速度稍微慢一点点，但对于你当前的学习和作业来说，结果的可控性要比微小的速度提升重要得多。

In [57]:
import torch.nn as nn
    # 定义一个模型
class HomeworkCNN(nn.Module):
    def __init__(self):
        super(HomeworkCNN, self).__init__()
        # 1. 卷积层：输入3通道，输出32通道，卷积核3x3,填充 = 0，步幅 = 1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=0)
        # 2. 激活函数（ReLU）,
        self.relu = nn.ReLU()
        # 3. 池化层：2x2,
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # 4. 展平后接全连接层,
        self.fc1 = nn.Linear(32 * 99 * 99, 64)   # 注意：200-3+1=198，池化后=99,
        self.fc2 = nn.Linear(64, 1)
        # 5. 输出层激活函数（Sigmoid）,
        #self.sigmoid = nn.Sigmoid()

# 前向传播函数
    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # # 输入图片 -> 卷积 -> 激活 -> 池化，提取出基础特征,
        x = x.view(x.size(0), -1)               # 展平，把二维的特征图“拉直”成一维向量，准备输入到全连接层,
        x = self.relu(self.fc1(x))              # 全连接1 -> ReLU   对特征进行非线性组合,
        #x = self.sigmoid(self.fc2(x))           # 全连接2 -> Sigmoid   输出一个0-1之间的概率值,
        x = self.fc2(x)  # ← 直接输出，不加Sigmoid
        return x

### 反向传播（Backward）：根据预测结果与真实标签的误差，从输出层反向计算梯度，更新网络权重。这个过程由 PyTorch 的自动求导机制（autograd）自动完成，不需要我们手动编写。

# Question 1
Which loss function you will use?

nn.BCEWithLogitsLoss()



# Question 2
20073473
### “参数量” 
和现在大家讨论大语言模型（LLM）时说的“参数量”（如 7B、70B、175B），本质上是一个概念——都是指神经网络中所有需要被训练和学习的权重（weights）和偏置（biases）的总个数。  
这个数字直接反映了模型的复杂度和容量。参数量越大，模型理论上能学习的信息就越多，能力也越强，但同时需要更多的计算资源和数据来训练。

In [58]:
model = HomeworkCNN()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}B")

# 也可以查看可训练参数和不可训练参数
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
non_trainable_params = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"Trainable: {trainable_params}, Non-trainable: {non_trainable_params}")

Total parameters: 20073473B
Trainable: 20073473, Non-trainable: 0


In [59]:
from torchvision import transforms    #计算机视觉任务的扩展包

train_transforms = transforms.Compose([ # 作用：将多个数据预处理步骤组合（串联） 成一个流水线。
    transforms.Resize((200, 200)),  #将输入图片缩放到 200×200 像素。
    transforms.ToTensor(),  #将 PIL 图片（或 NumPy 数组）转换为 PyTorch 张量（Tensor）。
    transforms.Normalize(   # 对张量进行标准化（Standardization），output = (input - mean) / std
        mean=[0.485, 0.456, 0.406],  #是 ImageNet 数据集的统计值，是深度学习社区约定俗成的标准
        std=[0.229, 0.224, 0.225]    # 同上
    ) # ImageNet normalization
])

In [60]:
# 开始训练模型
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os

In [61]:
# ============================================
# 1. 设置设备
# ============================================

In [62]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [63]:
# ============================================
# 2. 数据预处理（你已经定义好了 train_transforms）
# ============================================
# 验证集和测试集使用相同的预处理（不包含数据增强）

In [64]:
# 第一次执行完训练后，做数据增强
train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.RandomRotation(degrees=10),                 # 随机旋转 ±50 度
    # transforms.RandomResizedCrop(
    #    200,                                 # 裁剪后的大小
    #    scale=(0.9, 1.0),                    # 裁剪区域在原图的比例范围
    #    ratio=(0.9, 1.1)                     # 宽高比范围
    #),
    transforms.RandomHorizontalFlip(p=0.5),       # 随机水平翻转（50% 概率）
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [65]:
# 验证集使用完全相同的预处理
val_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [66]:
# ============================================
# 3. 加载数据集
# ============================================
data_dir = './data' 

train_dataset = datasets.ImageFolder(
    root=f'{data_dir}/train',
    transform=train_transforms
)

validation_dataset = datasets.ImageFolder(
    root=f'{data_dir}/test',  # 用 test 作为验证集
    transform=val_transforms
)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(validation_dataset)}")
print(f"Classes: {train_dataset.classes}")  #  ['curly', 'straight']


Training samples: 800
Validation samples: 201
Classes: ['curly', 'straight']


In [67]:
# ============================================
# 4. 创建 DataLoader
# ============================================
batch_size = 20  

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,      
    num_workers=0,  # （加载子进程数）0意味着不启用多进程
    pin_memory=True  # （锁定内存）当数据位于锁页内存时，从 CPU 到 GPU 的传输速度会快得多。
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=batch_size,
    shuffle=False,     
    num_workers=0,  
    pin_memory=True
)

print(f"Train loader: {len(train_loader)} batches")
print(f"Validation loader: {len(validation_loader)} batches")


Train loader: 40 batches
Validation loader: 11 batches


In [68]:
# ============================================
# 5. 初始化模型
# ============================================
model = HomeworkCNN().to(device)

In [69]:
# ============================================
# 6. 损失函数（二分类）
# ============================================
criterion = nn.BCEWithLogitsLoss()

In [70]:
# ============================================
# 7. 优化器
# ============================================
optimizer = optim.SGD(
    model.parameters(),
    lr=0.002,
    momentum=0.8
)

In [71]:
# 优化器调整 换用Adam（自适应学习率，更鲁棒）
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [72]:
# ============================================
# 8. 打印确认信息
# ============================================
print(f"\n{'='*50}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {batch_size}")
print(f"Epochs: {num_epochs}")
print(f"Train shuffle: True")
print(f"Validation shuffle: False")
print(f"{'='*50}\n")


Model parameters: 20,073,473
Batch size: 20
Epochs: 10
Train shuffle: True
Validation shuffle: False



In [73]:
# ============================================
# 9. 现在运行训练循环（alexey提供的代码）
# ============================================

## PyTorch 训练循环的标准模板代码

In [75]:
num_epochs = 20
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/20, Loss: 0.3354, Acc: 0.8512, Val Loss: 0.8553, Val Acc: 0.6567
Epoch 2/20, Loss: 0.3050, Acc: 0.8725, Val Loss: 0.5517, Val Acc: 0.7562
Epoch 3/20, Loss: 0.2654, Acc: 0.8925, Val Loss: 0.5363, Val Acc: 0.7861
Epoch 4/20, Loss: 0.2960, Acc: 0.8788, Val Loss: 0.6171, Val Acc: 0.7413
Epoch 5/20, Loss: 0.2560, Acc: 0.8888, Val Loss: 0.8182, Val Acc: 0.7065
Epoch 6/20, Loss: 0.2471, Acc: 0.9075, Val Loss: 0.6605, Val Acc: 0.7413
Epoch 7/20, Loss: 0.2696, Acc: 0.8938, Val Loss: 0.5370, Val Acc: 0.7811
Epoch 8/20, Loss: 0.2813, Acc: 0.8750, Val Loss: 0.5692, Val Acc: 0.7910
Epoch 9/20, Loss: 0.2016, Acc: 0.9163, Val Loss: 0.5915, Val Acc: 0.7761
Epoch 10/20, Loss: 0.2014, Acc: 0.9213, Val Loss: 0.6233, Val Acc: 0.7662
Epoch 11/20, Loss: 0.1804, Acc: 0.9275, Val Loss: 0.5821, Val Acc: 0.7861
Epoch 12/20, Loss: 0.1856, Acc: 0.9250, Val Loss: 0.9030, Val Acc: 0.6866
Epoch 13/20, Loss: 0.1804, Acc: 0.9275, Val Loss: 0.5605, Val Acc: 0.7861
Epoch 14/20, Loss: 0.1448, Acc: 0.9350, Val Los

# Question 3

What is the median of training accuracy for all the epochs for this model?

0.40


# Question 4
What is the standard deviation of training loss for all the epochs for this model?

0.078


In [ ]:
# Question 5
Let's train our model for 10 more epochs using the same code as previously.

Note: make sure you don't re-create the model. we want to continue training the model we already started training.

What is the mean of test loss for all the epochs for the model trained with augmentations?

0.88


In [ ]:
Question 6
What's the average of test accuracy for the last 5 epochs (from 6 to 10) for the model trained with augmentations?

0.68
